In [ ]:
# Imports & Configuration
from pathlib import Path

import duckdb
import polars as pl
from great_tables import GT

# Change only this path
SOURCE = Path("../../raw_data/raw_orders_sk.csv").resolve().as_posix()

# DuckDB connection
con = duckdb.connect()

In [ ]:
# ============================================================
# Functions
# ============================================================

def show_sql(query: str, max_rows: int = 100):
    """Execute DuckDB SQL and display results using Great Tables."""
    result = con.sql(query).limit(max_rows)

    df = pl.DataFrame(
        result.fetchall(),
        schema=result.columns,
        orient="row"
    )

    display(GT(df))

EXPECTED_CURRENCY = {
    "cz": "CZK",
    "sk": "EUR",
    "hu": "HUF"
}[Path(SOURCE).stem.split("_")[-1]]

In [ ]:
# 01 | Source Overview

show_sql(f"""
    SUMMARIZE
    SELECT *
    FROM read_csv('{SOURCE}', all_varchar = true)
""")

In [ ]:
# 02 | Data Quality Checks

show_sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(DISTINCT line_id) AS duplicate_rows,

        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(order_id), '') IS NULL
               OR NULLIF(TRIM(line_id), '') IS NULL
               OR NULLIF(TRIM(sku), '') IS NULL
               OR NULLIF(TRIM(created_at), '') IS NULL
               OR NULLIF(TRIM(exported_at), '') IS NULL
        ) AS missing_required,

        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(customer_id), '') IS NULL
        ) AS missing_customer,

        COUNT(*) FILTER (
            WHERE cancelled_at IS NOT NULL
        ) AS cancelled_rows,

        COUNT(*) FILTER (
            WHERE TRY_CAST(quantity AS INTEGER) IS NULL
               OR TRY_CAST(quantity AS INTEGER) <= 0
               OR TRY_CAST(price_with_vat AS DECIMAL(18,2)) IS NULL
               OR TRY_CAST(price_with_vat AS DECIMAL(18,2)) < 0
               OR TRY_CAST(discount_with_vat AS DECIMAL(18,2)) IS NULL
               OR TRY_CAST(discount_with_vat AS DECIMAL(18,2)) < 0
        ) AS invalid_numeric,

        COUNT(*) FILTER (
            WHERE currency IS DISTINCT FROM '{EXPECTED_CURRENCY}'
        ) AS invalid_currency,

        COUNT(*) FILTER (
            WHERE TRY_CAST(created_at AS TIMESTAMPTZ) IS NULL
               OR TRY_CAST(exported_at AS TIMESTAMP) IS NULL
               OR (
                   NULLIF(TRIM(cancelled_at), '') IS NOT NULL
                   AND TRY_CAST(cancelled_at AS TIMESTAMPTZ) IS NULL
               )
        ) AS invalid_timestamps

    FROM read_csv('{SOURCE}', all_varchar = true)
""")

In [ ]:
# 03 | Duplicate Investigation

show_sql(f"""
    WITH versions AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY line_id
                ORDER BY exported_at DESC
            ) AS rn
        FROM read_csv(
            '{SOURCE}',
            all_varchar = true
        )
    )
    SELECT
        COUNT(*) AS duplicated_lines,
        COUNT(*) FILTER (
            WHERE old.cancelled_at IS DISTINCT FROM new.cancelled_at
        ) AS changed_cancellations,
        COUNT(*) FILTER (
            WHERE old.quantity IS DISTINCT FROM new.quantity
               OR old.price_with_vat IS DISTINCT FROM new.price_with_vat
               OR old.discount_with_vat IS DISTINCT FROM new.discount_with_vat
        ) AS changed_financial_values
    FROM versions old
    JOIN versions new
        ON old.line_id = new.line_id
       AND old.rn = 2
       AND new.rn = 1
""")

In [ ]:
# 04 | Business Validity

show_sql(f"""
    SELECT
        COUNT(*) FILTER (
            WHERE TRIM(order_id) = ''
               OR TRIM(line_id) = ''
               OR TRIM(sku) = ''
               OR TRIM(currency) = ''
        ) AS empty_required,

        COUNT(*) FILTER (
            WHERE TRY_CAST(discount_with_vat AS DECIMAL(18,2))
                > TRY_CAST(price_with_vat AS DECIMAL(18,2))
                * TRY_CAST(quantity AS INTEGER)
        ) AS excessive_discount,

        COUNT(*) FILTER (
            WHERE cancelled_at IS NOT NULL
              AND TRY_CAST(cancelled_at AS TIMESTAMPTZ)
                < TRY_CAST(created_at AS TIMESTAMPTZ)
        ) AS invalid_cancellation,

        COUNT(DISTINCT order_id) AS unique_orders,
        COUNT(DISTINCT sku) AS unique_skus

    FROM read_csv('{SOURCE}', all_varchar = true)
""")

In [ ]:
# 05 | Schema

show_sql(f"""

    DESCRIBE
    SELECT *
    FROM read_csv_auto('{SOURCE}')
""")

In [ ]:
# 06 | Sample Data

show_sql(f"""
    SELECT *
    FROM read_csv('{SOURCE}', all_varchar = true)
    LIMIT 5
""")

| Area | Finding | Recommendation |
|---|---|---|
| Duplicates | 263 duplicate rows, including 118 cancellation updates | Retain the latest export per `line_id` |
| Missing values | 375 missing customer IDs | Preserve NULL values |
| Cancellations | 152 cancelled rows in raw data | Handle in downstream calculations |
| Data validity | No invalid numeric values, timestamps or business values detected | Apply appropriate data types |
| Currency | EUR | Preserve source currency until conversion |

**Source overview:** 5,439 rows, 3,356 unique orders and 32 unique SKUs.

**Expected rows after deduplication:** 5,176.

**Pending checks:** SKU integrity against the product catalog and FX rate coverage.